## 文檔摘要
 摘要：本程式主要負責將 TEJ 已處理過的資料表進行整合與清洗，並移除不必要的系統資訊檔案，最終結合所有 parquet 檔案為單一 DataFrame。隨後，程式會透過 FinMind API 取得台灣股票市場日曆資訊，並依指定區間過濾可用交易日。此處理流程有助於後續量化分析與資料庫建置的前置準備。

In [ ]:
import pandas as pd 
import os 
ROOT_DIR = '/home/newt/quant_project'
tej_data_path = os.path.join(ROOT_DIR, 'TEJ_processed_data')
def drop_zone_identifier(folder):
    # 刪除所有 Zone.Identifier 文件
    for file_path in os.listdir(folder):
        if file_path.endswith('.Identifier'):
            print(file_path)
            try:
                os.remove(os.path.join(folder, file_path))
                print(f"已刪除: {file_path}")
            except Exception as e:
                print(f"無法刪除 {file_path}: {e}")
new_data_folder = os.path.join(ROOT_DIR, 'zip_files_tradable_disposed' )
drop_zone_identifier(new_data_folder)

In [40]:
df_list = []
for file_path in os.listdir(tej_data_path):
    if file_path.endswith('.parquet'):
        df_list.append(pd.read_parquet(os.path.join(tej_data_path, file_path)))
df = pd.concat(df_list)
# 確保日期格式為 datetime（不轉換為字串）
df['date'] = pd.to_datetime(df['date'], errors='coerce').dt.normalize().dt.tz_localize(None)
df.head()

,instrument,date,open,high,low,close,volume,amount,return_day,turnover,...,PB_Ratio_TSE,PB_Ratio_TEJ,ps_ratio_tej,div_yield_tse,spread,adj_factor_price,adj_factor_volume,exchange,subindustry,industry
0,1264,2015-04-09,220.0,229.0,220.0,220.0,1518.0,338045.0,14.5833,4.9556,...,4.42,5.28,2.21,3.64,NaN,1.0,1.0,OTC,12Z 其他食品,M1200 食品工業
1,1264,2015-04-10,224.0,225.0,213.0,214.0,561.0,122686.0,-2.7273,1.8314,...,4.30,5.14,2.15,3.74,5.4545,1.0,1.0,OTC,12Z 其他食品,M1200 食品工業
2,1264,2015-04-13,215.0,218.0,211.0,212.0,281.0,60284.0,-0.9346,0.9173,...,4.26,5.09,2.13,3.77,3.2710,1.0,1.0,OTC,12Z 其他食品,M1200 食品工業
3,1264,2015-04-14,214.0,214.5,208.5,213.5,207.0,43938.0,0.7075,0.6758,...,4.29,5.13,2.15,3.75,2.8302,1.0,1.0,OTC,12Z 其他食品,M1200 食品工業
4,1264,2015-04-15,214.0,224.5,214.0,224.5,395.0,87136.0,5.1522,1.2895,...,4.51,5.39,2.26,3.56,4.9180,1.0,1.0,OTC,12Z 其他食品,M1200 食品工業


In [41]:
# 檢查原始 df（concat 後、尚未與 df_new_data merge）是否有 (date, instrument) 重複
n = len(df)
dup = df.duplicated(subset=['date', 'instrument'], keep=False)
n_dup = dup.sum()
print(f'原始 df 總筆數: {n}')
print(f'(date, instrument) 重複的筆數: {n_dup}')
print(f'唯一 (date, instrument) 數: {df.drop_duplicates(subset=["date", "instrument"]).shape[0]}')
if n_dup > 0:
    dup_pairs = df[dup][['date', 'instrument']].drop_duplicates()
    print(f'重複的 (date, instrument) 組數: {len(dup_pairs)}')
    print('重複組合樣本:')
    display(dup_pairs.head(10))

原始 df 總筆數: 8855432
(date, instrument) 重複的筆數: 0
唯一 (date, instrument) 數: 8855432


In [ ]:
# code = '2505'
# df_code = pd.read_parquet(os.path.join(tej_data_path, f'{code}_processed.parquet'))
# n = len(df_code)
# dup_key = df_code.duplicated(subset=['date', 'instrument'], keep=False)
# n_dup = dup_key.sum()
# print(f'原始 df 總筆數: {n}')
# print(f'(date, instrument) 重複的筆數: {n_dup}')
# print(f'唯一 (date, instrument) 數: {df_code.drop_duplicates(subset=["date", "instrument"]).shape[0]}')

# def _row_eq(v0, v1):
#     """兩值視為相同：相等或皆為 NaN"""
#     return (v0 == v1) or (pd.isna(v0) and pd.isna(v1))

# if n_dup > 0:
#     df_dup = df_code[dup_key].copy()
#     df_dup['_date_key'] = pd.to_datetime(df_dup['date'], errors='coerce').dt.normalize()
#     df_dup['_inst_key'] = df_dup['instrument'].astype(str)
#     data_cols = [c for c in df_dup.columns if not c.startswith('_')]
#     all_identical = []
#     has_difference_list = []
#     for (d_key, i_key), group in df_dup.groupby(['_date_key', '_inst_key']):
#         diff_cols = []
#         for c in data_cols:
#             v0 = group[c].iloc[0]
#             for r in range(1, len(group)):
#                 if not _row_eq(v0, group[c].iloc[r]):
#                     diff_cols.append(c)
#                     break
#         if not diff_cols:
#             all_identical.append((d_key, i_key))
#         else:
#             has_difference_list.append((d_key, i_key, diff_cols, group))
#     print(f'重複的 (date, instrument) 組數: {len(all_identical) + len(has_difference_list)}')
#     if all_identical:
#         print(f'所有欄位值皆相同的重複組數: {len(all_identical)}（可安全 drop_duplicates）')
#     if has_difference_list:
#         print(f'欄位值「不完全相同」的組數: {len(has_difference_list)}')
#         for d_key, i_key, diff_cols, group in has_difference_list:
#             print(f'  date={d_key}, instrument={i_key}')
#             print(f'  不同的欄位: {diff_cols}')
#             display(group.drop(columns=['_date_key', '_inst_key']))
#     print('重複組合樣本:')
#     display(df_dup[['_date_key', '_inst_key']].drop_duplicates().rename(columns={'_date_key': 'date', '_inst_key': 'instrument'}).head(10))
#df_code = df_code.drop_duplicates()
#df_code.to_parquet(os.path.join(tej_data_path, f'{code}_processed.parquet'))

原始 df 總筆數: 6444
(date, instrument) 重複的筆數: 0
唯一 (date, instrument) 數: 6444


In [ ]:
# 人工逐一檢查
# g = df_code[(df_code['date'].dt.normalize() == pd.Timestamp('2021-09-13').normalize()) & (df_code['instrument'].astype(str) == '2236')]
# for c in g.columns:
#     v0, v1 = g[c].iloc[0], g[c].iloc[1]
#     eq = (v0 == v1) or (pd.isna(v0) and pd.isna(v1))
    
#     print(f'{c}:')
#     print(f'  row0: {repr(v0)}')
#     print(f'  row1: {repr(v1)}')
#     print(f'  相同: {eq}')
#     print()

In [42]:
# 抓出所有 date unique 然後找出最小的五個
unique_dates = df['date'].unique()
min_dates = sorted(unique_dates)[:5]
min_dates
# 抓出所有 date unique 然後找出最小的五個
unique_dates = df['date'].unique()
min_dates = sorted(unique_dates)[:5]
min_dates

[numpy.datetime64('2000-01-03T00:00:00.000000000'),
 numpy.datetime64('2000-01-04T00:00:00.000000000'),
 numpy.datetime64('2000-01-05T00:00:00.000000000'),
 numpy.datetime64('2000-01-06T00:00:00.000000000'),
 numpy.datetime64('2000-01-07T00:00:00.000000000')]

In [43]:
new_df_list = []
for file_path in os.listdir(new_data_folder):
    if file_path.endswith('.parquet'):
        new_df_list.append(pd.read_parquet(os.path.join(new_data_folder, file_path)))
df_new_data = pd.concat(new_df_list)
# 確保 df_new_data 的日期格式一致
df_new_data['date'] = pd.to_datetime(df_new_data['date'], errors='coerce').dt.normalize().dt.tz_localize(None)

In [44]:
from FinMind.data import DataLoader
from config import  FINMIND_TOKEN
import requests
api = DataLoader()
api.login_by_token(api_token=FINMIND_TOKEN)

url = "https://api.finmindtrade.com/api/v4/data"
headers = {"Authorization": f"Bearer {FINMIND_TOKEN}"}
parameter = {
    "dataset": "TaiwanStockTradingDate",
}
resp = requests.get(url, headers=headers, params=parameter)
calendar = resp.json()
calendar = pd.DataFrame(calendar["data"])
# 方法 1：使用日期範圍過濾
calendar = calendar[
    (pd.to_datetime(calendar['date']) >= '2000-01-01') & 
    (pd.to_datetime(calendar['date']) <= '2026-01-23')
]
calendar['date'] = pd.to_datetime(calendar['date'], errors='coerce').dt.normalize().dt.tz_localize(None)
display(calendar.head())
display(calendar.tail())

2026-02-14 02:34:03.913 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success
2026-02-14 02:34:03.959 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success


,date
266,2000-01-04
267,2000-01-05
268,2000-01-06
269,2000-01-07
270,2000-01-10


,date
6705,2026-01-19
6706,2026-01-20
6707,2026-01-21
6708,2026-01-22
6709,2026-01-23


In [6]:
# # 詳細診斷：檢查日期格式和實際內容
# print("="*60)
# print("詳細日期診斷")
# print("="*60)

# # 1. 檢查 df 的實際日期範圍和格式
# df_sorted_dates = sorted(df['date'].unique())
# print(f"\ndf 日期範圍:")
# print(f"  最小日期: {df_sorted_dates[0]}")
# print(f"  最大日期: {df_sorted_dates[-1]}")
# print(f"  總日期數: {len(df_sorted_dates)}")
# print(f"  前5個日期: {df_sorted_dates[:5]}")
# print(f"  前5個日期類型: {[type(d) for d in df_sorted_dates[:5]]}")

# # 2. 檢查 calendar 的日期範圍和格式
# calendar_sorted_dates = sorted(calendar['date'].unique())
# print(f"\ncalendar 日期範圍:")
# print(f"  最小日期: {calendar_sorted_dates[0]}")
# print(f"  最大日期: {calendar_sorted_dates[-1]}")
# print(f"  總日期數: {len(calendar_sorted_dates)}")
# print(f"  前5個日期: {calendar_sorted_dates[:5]}")
# print(f"  前5個日期類型: {[type(d) for d in calendar_sorted_dates[:5]]}")

# # 3. 檢查特定日期（2000-01-03, 2000-01-04）是否真的存在
# test_dates = [
#     pd.Timestamp('2000-01-03'),
#     pd.Timestamp('2000-01-04'),
#     pd.Timestamp('2000-01-05'),
# ]

# print(f"\n驗證特定日期:")
# for test_date in test_dates:
#     # 標準化日期以便比較
#     test_date_normalized = pd.Timestamp(test_date).normalize()
    
#     # 檢查在 df 中
#     in_df = test_date_normalized in set(df['date'].unique())
#     # 檢查在 calendar 中
#     in_calendar = test_date_normalized in set(calendar['date'].unique())
    
#     print(f"  {test_date_normalized.strftime('%Y-%m-%d')}:")
#     print(f"    在 df 中: {in_df}")
#     print(f"    在 calendar 中: {in_calendar}")
    
#     # 如果不在 df 中，檢查是否有類似的日期（可能是格式問題）
#     if not in_df:
#         # 檢查是否有相同日期的不同格式
#         df_matches = df[df['date'].dt.date == test_date_normalized.date()]
#         if len(df_matches) > 0:
#             print(f"    警告：df 中有相同日期的記錄，但格式可能不同")
#             print(f"    找到的記錄數: {len(df_matches)}")
#             print(f"    日期值: {df_matches['date'].iloc[0]}")
#             print(f"    日期值類型: {type(df_matches['date'].iloc[0])}")

# # 4. 檢查集合比對的問題
# print(f"\n集合比對診斷:")
# df_dates_set = set(df['date'].unique())
# calendar_dates_set = set(calendar['date'].unique())

# # 檢查日期對象的哈希值是否一致
# sample_df_date = list(df_dates_set)[0]
# sample_calendar_date = list(calendar_dates_set)[0]

# print(f"  df 中樣本日期: {sample_df_date}, 類型: {type(sample_df_date)}, 哈希: {hash(sample_df_date)}")
# print(f"  calendar 中樣本日期: {sample_calendar_date}, 類型: {type(sample_calendar_date)}, 哈希: {hash(sample_calendar_date)}")

# # 檢查 2000-01-04 是否在兩個集合中
# test_date_20000104 = pd.Timestamp('2000-01-04').normalize()
# print(f"\n  2000-01-04 檢查:")
# print(f"    標準化日期: {test_date_20000104}")
# print(f"    在 df_dates_set 中: {test_date_20000104 in df_dates_set}")
# print(f"    在 calendar_dates_set 中: {test_date_20000104 in calendar_dates_set}")

# # 如果不在 df_dates_set 中，檢查是否有接近的日期
# if test_date_20000104 not in df_dates_set:
#     # 檢查 df 中是否有 2000-01-04 的記錄（使用日期部分比較）
#     df_20000104 = df[df['date'].dt.date == test_date_20000104.date()]
#     if len(df_20000104) > 0:
#         print(f"    警告：df 中有 2000-01-04 的記錄，但集合中沒有！")
#         print(f"    記錄數: {len(df_20000104)}")
#         print(f"    實際日期值: {df_20000104['date'].iloc[0]}")
#         print(f"    日期值類型: {type(df_20000104['date'].iloc[0])}")
#         print(f"    日期值是否等於 test_date: {df_20000104['date'].iloc[0] == test_date_20000104}")
#         print(f"    日期值是否在集合中: {df_20000104['date'].iloc[0] in df_dates_set}")

In [45]:


# 獲取唯一的日期集合（使用 datetime 格式）
df_dates = set(df['date'].unique())
calendar_dates = set(calendar['date'].unique())

# 找出差異
# 1. 在 df 中但不在 calendar 中的日期（可能是非交易日但有數據）
dates_in_df_not_in_calendar = df_dates - calendar_dates

# 2. 在 calendar 中但不在 df 中的日期（交易日但沒有數據）
dates_in_calendar_not_in_df = calendar_dates - df_dates

# 3. 兩者都有的日期
dates_in_both = df_dates & calendar_dates

# 顯示結果
print(f"df 中的總日期數: {len(df_dates)}")
print(f"calendar 中的總日期數: {len(calendar_dates)}")
print(f"兩者都有的日期數: {len(dates_in_both)}")
print(f"\n在 df 中但不在 calendar 中的日期數: {len(dates_in_df_not_in_calendar)}")
print(f"在 calendar 中但不在 df 中的日期數: {len(dates_in_calendar_not_in_df)}")

# 顯示具體的日期（顯示時可以格式化為字串，但集合中保持 datetime）
if dates_in_df_not_in_calendar:
    print(f"\n在 df 中但不在 calendar 中的日期（前20個）:")
    print(len(dates_in_df_not_in_calendar))
    # 排序並格式化顯示
    sorted_dates = sorted(list(dates_in_df_not_in_calendar))[:20]
    print([pd.Timestamp(date).strftime('%Y-%m-%d') for date in sorted_dates])
    
if dates_in_calendar_not_in_df:
    print(f"\n在 calendar 中但不在 df 中的日期（前20個）:")
    print(len(dates_in_calendar_not_in_df))
    # 排序並格式化顯示
    sorted_dates = sorted(list(dates_in_calendar_not_in_df))[:20]
    print([pd.Timestamp(date).strftime('%Y-%m-%d') for date in sorted_dates])

# ========== df_new_data 與 calendar 的比對 ==========
print("\n" + "="*50)
print("df_new_data 與 calendar 的比對")
print("="*50)



# 獲取唯一的日期集合
df_new_data_dates = set(df_new_data['date'].unique())

# 找出差異
# 1. 在 df_new_data 中但不在 calendar 中的日期（可能是非交易日但有數據）
dates_in_df_new_data_not_in_calendar = df_new_data_dates - calendar_dates

# 2. 在 calendar 中但不在 df_new_data 中的日期（交易日但沒有數據）
dates_in_calendar_not_in_df_new_data = calendar_dates - df_new_data_dates

# 3. 兩者都有的日期
dates_in_both_new_data = df_new_data_dates & calendar_dates

# 顯示結果
print(f"df_new_data 中的總日期數: {len(df_new_data_dates)}")
print(f"calendar 中的總日期數: {len(calendar_dates)}")
print(f"兩者都有的日期數: {len(dates_in_both_new_data)}")
print(f"\n在 df_new_data 中但不在 calendar 中的日期數: {len(dates_in_df_new_data_not_in_calendar)}")
print(f"在 calendar 中但不在 df_new_data 中的日期數: {len(dates_in_calendar_not_in_df_new_data)}")

# 顯示具體的日期
if dates_in_df_new_data_not_in_calendar:
    print(f"\n在 df_new_data 中但不在 calendar 中的日期（前20個）:")
    print(len(dates_in_df_new_data_not_in_calendar))
    print(sorted(list(dates_in_df_new_data_not_in_calendar))[:20])
    
if dates_in_calendar_not_in_df_new_data:
    print(f"\n在 calendar 中但不在 df_new_data 中的日期（前20個）:")
    print(len(dates_in_calendar_not_in_df_new_data))
    print(sorted(list(dates_in_calendar_not_in_df_new_data))[:20])

df 中的總日期數: 6929
calendar 中的總日期數: 6444
兩者都有的日期數: 6444

在 df 中但不在 calendar 中的日期數: 485
在 calendar 中但不在 df 中的日期數: 0

在 df 中但不在 calendar 中的日期（前20個）:
485
['2000-01-03', '2000-07-01', '2000-11-11', '2000-12-31', '2001-01-01', '2001-02-17', '2001-02-18', '2001-02-28', '2001-03-31', '2001-05-05', '2001-07-29', '2001-09-01', '2001-09-12', '2001-09-17', '2001-10-01', '2001-10-06', '2001-11-11', '2001-12-01', '2001-12-15', '2001-12-22']

df_new_data 與 calendar 的比對
df_new_data 中的總日期數: 6444
calendar 中的總日期數: 6444
兩者都有的日期數: 6444

在 df_new_data 中但不在 calendar 中的日期數: 0
在 calendar 中但不在 df_new_data 中的日期數: 0


## 檢查日期匹配狀況 
TEJ數據會出現沒有發生交易的日期 多是紀錄其他事件的發生
### 處理順序
--先處理calender 中有數據 但是不在df 中的 ==> 這裡暫不做因為2026/01/23前沒有  

--再處理df 中有數據 但是不在先處理calender 中的

In [46]:
need_process_date = []
for date in list(dates_in_df_not_in_calendar):
    if df[df['date'] == date]['volume'].sum() > 0:
        need_process_date.append(date)
if len(need_process_date) == 0:
    print('沒有需要處理的日期')
    # 把dates_in_df_not_in_calendar這些日期從df拿走
    df = df[~df['date'].isin(dates_in_df_not_in_calendar)]

沒有需要處理的日期


In [47]:
need_process_date_new_data = []
for date in list(dates_in_df_new_data_not_in_calendar):
    if df_new_data[df_new_data['date'] == date]['volume'].sum() > 0:
        need_process_date_new_data.append(date)
if len(need_process_date_new_data) == 0:
    print('沒有需要處理的日期')
    # 把dates_in_df_not_in_calendar這些日期從df拿走
    df_new_data = df_new_data[~df_new_data['date'].isin(dates_in_df_new_data_not_in_calendar)]

沒有需要處理的日期


In [48]:


# 獲取唯一的日期集合（使用 datetime 格式）
df_dates = set(df['date'].unique())
calendar_dates = set(calendar['date'].unique())

# 找出差異
# 1. 在 df 中但不在 calendar 中的日期（可能是非交易日但有數據）
dates_in_df_not_in_calendar = df_dates - calendar_dates

# 2. 在 calendar 中但不在 df 中的日期（交易日但沒有數據）
dates_in_calendar_not_in_df = calendar_dates - df_dates

# 3. 兩者都有的日期
dates_in_both = df_dates & calendar_dates

# 顯示結果
print(f"df 中的總日期數: {len(df_dates)}")
print(f"calendar 中的總日期數: {len(calendar_dates)}")
print(f"兩者都有的日期數: {len(dates_in_both)}")
print(f"\n在 df 中但不在 calendar 中的日期數: {len(dates_in_df_not_in_calendar)}")
print(f"在 calendar 中但不在 df 中的日期數: {len(dates_in_calendar_not_in_df)}")

# 顯示具體的日期（顯示時可以格式化為字串，但集合中保持 datetime）
if dates_in_df_not_in_calendar:
    print(f"\n在 df 中但不在 calendar 中的日期（前20個）:")
    print(len(dates_in_df_not_in_calendar))
    # 排序並格式化顯示
    sorted_dates = sorted(list(dates_in_df_not_in_calendar))[:20]
    print([date.strftime('%Y-%m-%d') for date in sorted_dates])
    
if dates_in_calendar_not_in_df:
    print(f"\n在 calendar 中但不在 df 中的日期（前20個）:")
    print(len(dates_in_calendar_not_in_df))
    # 排序並格式化顯示
    sorted_dates = sorted(list(dates_in_calendar_not_in_df))[:20]
    print([date.strftime('%Y-%m-%d') for date in sorted_dates])

# ========== df_new_data 與 calendar 的比對 ==========
print("\n" + "="*50)
print("df_new_data 與 calendar 的比對")
print("="*50)



# 獲取唯一的日期集合
df_new_data_dates = set(df_new_data['date'].unique())

# 找出差異
# 1. 在 df_new_data 中但不在 calendar 中的日期（可能是非交易日但有數據）
dates_in_df_new_data_not_in_calendar = df_new_data_dates - calendar_dates

# 2. 在 calendar 中但不在 df_new_data 中的日期（交易日但沒有數據）
dates_in_calendar_not_in_df_new_data = calendar_dates - df_new_data_dates

# 3. 兩者都有的日期
dates_in_both_new_data = df_new_data_dates & calendar_dates

# 顯示結果
print(f"df_new_data 中的總日期數: {len(df_new_data_dates)}")
print(f"calendar 中的總日期數: {len(calendar_dates)}")
print(f"兩者都有的日期數: {len(dates_in_both_new_data)}")
print(f"\n在 df_new_data 中但不在 calendar 中的日期數: {len(dates_in_df_new_data_not_in_calendar)}")
print(f"在 calendar 中但不在 df_new_data 中的日期數: {len(dates_in_calendar_not_in_df_new_data)}")

# 顯示具體的日期
if dates_in_df_new_data_not_in_calendar:
    print(f"\n在 df_new_data 中但不在 calendar 中的日期（前20個）:")
    print(len(dates_in_df_new_data_not_in_calendar))
    print(sorted(list(dates_in_df_new_data_not_in_calendar))[:20])
    
if dates_in_calendar_not_in_df_new_data:
    print(f"\n在 calendar 中但不在 df_new_data 中的日期（前20個）:")
    print(len(dates_in_calendar_not_in_df_new_data))
    print(sorted(list(dates_in_calendar_not_in_df_new_data))[:20])

df 中的總日期數: 6444
calendar 中的總日期數: 6444
兩者都有的日期數: 6444

在 df 中但不在 calendar 中的日期數: 0
在 calendar 中但不在 df 中的日期數: 0

df_new_data 與 calendar 的比對
df_new_data 中的總日期數: 6444
calendar 中的總日期數: 6444
兩者都有的日期數: 6444

在 df_new_data 中但不在 calendar 中的日期數: 0
在 calendar 中但不在 df_new_data 中的日期數: 0


In [49]:
# 合併 df_new_data 到 df
print("="*60)
print("合併 df_new_data 到 df")
print("="*60)

# 1. 確保日期格式一致
df['date'] = pd.to_datetime(df['date'], errors='coerce').dt.normalize().dt.tz_localize(None)
df_new_data['date'] = pd.to_datetime(df_new_data['date'], errors='coerce').dt.normalize().dt.tz_localize(None)

# 2. 檢查 df 和 df_new_data 的欄位
print(f"\ndf 的欄位: {list(df.columns)}")
print(f"df_new_data 的欄位: {list(df_new_data.columns)}")

# 3. 檢查是否有重複的欄位（除了 date 和 instrument）
common_cols = set(df.columns) & set(df_new_data.columns)
print(f"\n共同欄位: {common_cols}")

# 4. 使用 merge 合併（使用 date 和 instrument 作為 key）
# 使用 left merge 保留 df 的所有記錄，並從 df_new_data 添加 is_disposed 和 is_tradable
df_merged = df.merge(
    df_new_data[['date', 'instrument', 'is_disposed', 'is_tradable']],
    on=['date', 'instrument'],
    how='left',  # 保留 df 的所有記錄
    suffixes=('', '_new')  # 如果有重複欄位，添加後綴
)

print(f"\n合併後的欄位: {list(df_merged.columns)}")
print(f"合併前 df 的記錄數: {len(df)}")
print(f"合併後 df_merged 的記錄數: {len(df_merged)}")

# 5. 檢查 is_disposed 和 is_tradable 的 NA 值
print("\n" + "="*60)
print("檢查 is_disposed 和 is_tradable 的 NA 值")
print("="*60)

# 檢查 NA 值的數量
na_is_disposed = df_merged['is_disposed'].isna().sum()
na_is_tradable = df_merged['is_tradable'].isna().sum()
total_records = len(df_merged)

print(f"\n總記錄數: {total_records}")
print(f"is_disposed 的 NA 值數量: {na_is_disposed} ({na_is_disposed/total_records*100:.2f}%)")
print(f"is_tradable 的 NA 值數量: {na_is_tradable} ({na_is_tradable/total_records*100:.2f}%)")

# 檢查兩個欄位都是 NA 的情況
both_na = df_merged['is_disposed'].isna() & df_merged['is_tradable'].isna()
print(f"兩個欄位都是 NA 的記錄數: {both_na.sum()} ({both_na.sum()/total_records*100:.2f}%)")

# 6. 顯示有 NA 值的記錄樣本
if na_is_disposed > 0 or na_is_tradable > 0:
    print("\n有 NA 值的記錄樣本（前10筆）:")
    na_mask = df_merged['is_disposed'].isna() | df_merged['is_tradable'].isna()
    na_samples = df_merged[na_mask][['date', 'instrument', 'is_disposed', 'is_tradable']].head(10)
    display(na_samples)
    
    # 檢查 NA 值的日期分布
    print("\nNA 值的日期分布（前10個日期）:")
    na_dates = df_merged[na_mask]['date'].value_counts().head(10)
    print(na_dates)

# 7. 檢查是否有重複的 date-instrument 組合
print("\n" + "="*60)
print("檢查重複的 date-instrument 組合")
print("="*60)
duplicates = df_merged.duplicated(subset=['date', 'instrument'], keep=False)
if duplicates.sum() > 0:
    print(f"發現重複的 date-instrument 組合: {duplicates.sum()} 筆記錄")
    print("\n重複記錄樣本（前10筆）:")
    display(df_merged[duplicates][['date', 'instrument', 'is_disposed', 'is_tradable']].head(10))
else:
    print("沒有發現重複的 date-instrument 組合")

#8. 將合併後的結果賦值回 df（如果需要）
df = df_merged.copy()

合併 df_new_data 到 df

df 的欄位: ['instrument', 'date', 'open', 'high', 'low', 'close', 'volume', 'amount', 'return_day', 'turnover', 'market_value_million', 'last_bid', 'last_ask', 'log_return_day', 'market_value_pct', 'trading_value_pct', 'numtrades', 'PE_ration_TSE', 'PE_ration_TEJ', 'PB_Ratio_TSE', 'PB_Ratio_TEJ', 'ps_ratio_tej', 'div_yield_tse', 'spread', 'adj_factor_price', 'adj_factor_volume', 'exchange', 'subindustry', 'industry']
df_new_data 的欄位: ['date', 'instrument', 'is_disposed', 'is_tradable']

共同欄位: {'date', 'instrument'}

合併後的欄位: ['instrument', 'date', 'open', 'high', 'low', 'close', 'volume', 'amount', 'return_day', 'turnover', 'market_value_million', 'last_bid', 'last_ask', 'log_return_day', 'market_value_pct', 'trading_value_pct', 'numtrades', 'PE_ration_TSE', 'PE_ration_TEJ', 'PB_Ratio_TSE', 'PB_Ratio_TEJ', 'ps_ratio_tej', 'div_yield_tse', 'spread', 'adj_factor_price', 'adj_factor_volume', 'exchange', 'subindustry', 'industry', 'is_disposed', 'is_tradable']
合併前 df 的記錄數:

In [58]:
from tqdm import tqdm
for code, code_df in tqdm(df.groupby('instrument')):
    code_list = code_df['instrument'].unique()
    if len(code_list) == 1:
        code = code_list[0]
        code_df.to_parquet(os.path.join(tej_data_path, f'{code}_processed.parquet'))
    else:
        print(f'{code} 有 {len(code_list)} 個 code')
    

100%|██████████| 2355/2355 [00:22<00:00, 102.51it/s]
